# YOLO Hangul Text Detection

한글 손글씨 **영역의 바운딩 박스**를 검출합니다. 글자를 읽거나 문자열로 변환하는 OCR은 아닙니다.

원본: `pod369/Image-processing-4/Hangeul.ipynb`. YOLOv8n, 100 epochs, 640px 설정을 유지합니다.
데이터셋과 가중치는 포함되어 있지 않습니다. 먼저 직접 준비한 `data.yaml`과 이미지/라벨 경로를 설정하세요.
새 학습은 아래 셀을 직접 실행했을 때만 수행됩니다. 원본의 출력은 `original_training_summary.txt`에 별도 표시했습니다.


In [ ]:
%pip install "ultralytics==8.3.234"


## Dataset path

Colab에서는 데이터셋을 `/content/hangul_dataset`에 준비하거나 Google Drive를 연결한 후 절대 경로를 지정합니다.
로컬 Jupyter에서는 PC의 데이터셋 절대 경로로 바꿉니다. `data.example.yaml`의 형식을 참고하세요.
학습/검증/시험은 다른 이미지로 나누고, 같은 원본 이미지의 변형도 서로 다른 분할에 섞지 않습니다.


In [ ]:
from pathlib import Path
import yaml
from ultralytics import YOLO

DATA_YAML = Path("/content/hangul_dataset/data.yaml")  # Change this path.
OUTPUT_DIR = Path("outputs").resolve()
SEED = 42

assert DATA_YAML.is_file(), f"Prepare your dataset YAML first: {DATA_YAML}"
config = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))
assert isinstance(config, dict) and config.get("train") and config.get("val")
assert len(config.get("names", [])) == 1, "Use one Hangul text-region class."


## Train

처음 실행하면 Ultralytics가 사전학습된 `yolov8n.pt`를 내려받습니다. GPU 사용은 런타임에서 직접 설정하세요.
결과는 Ultralytics의 실제 실행 폴더에 저장되고, 같은 이름이 있으면 새 번호가 붙을 수 있습니다.


In [ ]:
model = YOLO("yolov8n.pt")
model.train(data=str(DATA_YAML.resolve()), epochs=100, imgsz=640,
            batch=16, seed=SEED, deterministic=True, workers=2,
            project=str(OUTPUT_DIR), name="train")
RUN_DIR = Path(model.trainer.save_dir)
BEST_WEIGHTS = RUN_DIR / "weights" / "best.pt"
print("Run directory:", RUN_DIR)
print("Best weights:", BEST_WEIGHTS)


In [ ]:
from IPython.display import Image, display

results_image = RUN_DIR / "results.png"
if results_image.is_file():
    display(Image(filename=str(results_image)))


## Predict

`IMAGE_PATH`를 실제 시험용 이미지로 변경합니다. 출력은 탐지 영역이며, 텍스트 내용은 출력하지 않습니다.


In [ ]:
from PIL import Image as PILImage

IMAGE_PATH = Path("/content/hangul_dataset/test/images/example.jpg")  # Change this path.
assert IMAGE_PATH.is_file(), f"Choose an existing image: {IMAGE_PATH}"
detector = YOLO(str(BEST_WEIGHTS))
results = detector.predict(source=str(IMAGE_PATH), conf=0.25, imgsz=640,
                          save=True, save_txt=True, save_conf=True,
                          project=str(OUTPUT_DIR), name="predict")
for result in results:
    print("Text regions:", len(result.boxes), "Saved in:", result.save_dir)
    display(PILImage.fromarray(result.plot()[:, :, ::-1]))


## Optional held-out test evaluation

학습 및 모델 선택에 사용하지 않은 `test` 분할을 `data.yaml`에 추가한 경우에만 실행하세요.
실제 데이터 없이 성능 수치를 생성하거나 원본 검증 결과를 시험 결과로 사용하지 않습니다.


In [ ]:
RUN_TEST_EVALUATION = False  # Set True only after reserving a separate test split.
if RUN_TEST_EVALUATION:
    assert config.get("test"), "Add a separate test split to data.yaml."
    test_detector = YOLO(str(BEST_WEIGHTS))
    test_metrics = test_detector.val(data=str(DATA_YAML.resolve()), split="test", imgsz=640,
                                project=str(OUTPUT_DIR), name="test")
    print(test_metrics.results_dict)
